# Experiment 3: Breadth-First and Depth-First Search

**Course:** CSAI4014 – Algorithms for Intelligent Systems and Robotics  
**Experiment:** 3

Run all cells from top to bottom in a fresh kernel. The notebook creates a maze with a guaranteed route, runs BFS and DFS using one search function, validates the paths, and plots their expansion order.

## 1. Aim
To represent a maze as an implicit graph, to implement breadth-first and depth-first search as one function differing in a single line, and to compare them on the number of cells expanded and the length of the path returned.

## 2. Objectives
1. Model a grid as a graph whose edges are computed on demand rather than stored.
2. Implement a frontier, an explored set and a parent map.
3. Show that the only difference between the two searches is the end of the frontier used.
4. Reconstruct a path from the parent map and validate it.
5. Measure and explain the trade-off between cells expanded and path length.

## 3. Requirements
Python with NumPy and Matplotlib. `collections.deque` provides the frontier.

## 4. Theory
A search problem consists of a state space, a start state, a goal test and a successor function. The frontier contains discovered states that have not yet been expanded. The explored set contains states already expanded, and the parent map stores how each state was reached. Breadth-first search (BFS) removes the oldest frontier item (FIFO queue); with equal step costs, it guarantees a shortest path. Depth-first search (DFS) removes the newest item (LIFO stack), follows one branch deeply and does not guarantee a shortest path.

## 5. Create a maze and define neighbours

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random

# 1 = wall, 0 = open cell
H, W = 23, 35
maze = np.ones((H, W), dtype=int)
rng = random.Random(31)

# Randomized depth-first carving guarantees a connected route.
maze[1, 1] = 0
stack = [(1, 1)]
while stack:
    r, c = stack[-1]
    options = []
    for dr, dc in [(-2, 0), (2, 0), (0, -2), (0, 2)]:
        nr, nc = r + dr, c + dc
        if 1 <= nr < H-1 and 1 <= nc < W-1 and maze[nr, nc] == 1:
            options.append((nr, nc, dr, dc))
    if options:
        nr, nc, dr, dc = rng.choice(options)
        maze[r + dr//2, c + dc//2] = 0
        maze[nr, nc] = 0
        stack.append((nr, nc))
    else:
        stack.pop()

# Add some passages to create alternative routes.
for r in range(1, H-1):
    for c in range(1, W-1):
        if maze[r, c] == 1 and rng.random() < 0.16:
            maze[r, c] = 0

START, GOAL = (1, 1), (H-2, W-2)
print("Maze dimensions:", maze.shape)
print("Open cells:", int(np.sum(maze == 0)))

def neighbours(cell):
    r, c = cell
    for dr, dc in ((-1,0), (1,0), (0,-1), (0,1)):
        nr, nc = r + dr, c + dc
        # Check bounds before reading the grid.
        if 0 <= nr < H and 0 <= nc < W:
            if maze[nr, nc] == 0:
                yield (nr, nc)

plt.figure(figsize=(8, 5))
plt.imshow(maze, cmap="binary")
plt.scatter([START[1], GOAL[1]], [START[0], GOAL[0]], label="Start / Goal")
plt.title("Generated maze (1 = wall, 0 = open)")
plt.xlabel("Column index")
plt.ylabel("Row index")
plt.legend()
plt.show()


## 6. Implement BFS and DFS in one search function

In [ ]:
def search(start, goal, mode="bfs"):
    frontier = deque([start])
    parent = {start: None}  # also marks cells already discovered
    order = {}               # cell -> expansion sequence number

    while frontier:
        # The only algorithmic difference:
        node = frontier.popleft() if mode == "bfs" else frontier.pop()

        if node in order:
            continue
        order[node] = len(order)
        if node == goal:
            break

        for nxt in neighbours(node):
            if nxt not in parent:
                parent[nxt] = node
                frontier.append(nxt)

    # Reconstruct path by following parent links backwards.
    path, cur = [], goal
    if goal in parent:
        while cur is not None:
            path.append(cur)
            cur = parent[cur]

    return order, path[::-1]

bfs_order, bfs_path = search(START, GOAL, "bfs")
dfs_order, dfs_path = search(START, GOAL, "dfs")

print(f"BFS: {len(bfs_order)} cells expanded, path length {len(bfs_path)} cells")
print(f"DFS: {len(dfs_order)} cells expanded, path length {len(dfs_path)} cells")


## 7. Validate both paths

In [ ]:
def validate_path(path):
    if not path:
        return False
    # Every cell must be open and inside the maze.
    for r, c in path:
        if not (0 <= r < H and 0 <= c < W):
            return False
        if maze[r, c] != 0:
            return False
    # Each consecutive pair must be orthogonally adjacent.
    for (r1, c1), (r2, c2) in zip(path, path[1:]):
        if abs(r1-r2) + abs(c1-c2) != 1:
            return False
    return path[0] == START and path[-1] == GOAL

print("BFS path valid:", validate_path(bfs_path))
print("DFS path valid:", validate_path(dfs_path))


## 8. Plot expansion order and returned paths

In [ ]:
def plot_search(order, path, mode):
    grid = np.full(maze.shape, np.nan)
    for cell, step in order.items():
        grid[cell] = step

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.imshow(maze, cmap="binary")
    ax.imshow(grid, cmap="viridis", alpha=0.72)
    if path:
        ax.plot([c for r, c in path], [r for r, c in path],
                linewidth=2, label="Returned path")
    ax.scatter([START[1], GOAL[1]], [START[0], GOAL[0]],
               marker="o", s=55, label="Start / Goal")
    ax.set_title(f"{mode}: {len(order)} expanded, path length {len(path)}")
    ax.set_xlabel("Column index")
    ax.set_ylabel("Row index")
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_search(bfs_order, bfs_path, "BFS")
plot_search(dfs_order, dfs_path, "DFS")


## 9. Results and observations
Record maze dimensions, number of open cells, cells expanded and path length for both BFS and DFS, and the validation result for each path. The notebook prints these values after running.

## 10. Discussion
BFS explores states in increasing distance from the start. Since every move has equal cost, the first path found is a shortest path. DFS follows the most recently discovered branch and can reach the goal along a longer route. The number of cells expanded and the DFS path length depend on the maze layout; they are not fixed properties of either algorithm. Both returned paths must pass validation.

## 11. Applications
BFS underlies shortest-hop routing in a network and wavefront planning on occupancy grids. DFS is used in topological sorting, cycle detection and maze carving. Both are basic search methods used as foundations for more advanced planning algorithms.

## 12. Conclusion
The maze was represented as an implicit graph and searched using BFS and DFS implemented in one function. The algorithms differ in which end of the frontier is removed. The returned paths were reconstructed using a parent map and checked for valid, adjacent, wall-free steps.

## 13. Viva questions
1. Your DFS path is longer than your BFS path. What can you conclude, and what can you not?
2. Why must the bounds check come before the wall check?
3. Why is a cell marked when it is enqueued rather than when it is expanded?
4. State the space complexity of each algorithm and say which you would choose on a memory-constrained robot.
5. If diagonal moves were allowed, would BFS still return a shortest path? Justify.

## Source
CSAI4014 Algorithms for Intelligent Systems and Robotics Laboratory Manual, Experiment 3: Breadth-first and depth-first search, Chapter 4, pages 20–23.